# 04 — Modeling

Each candidate is a single sklearn `Pipeline` of the form `[preprocess → model]`. The same `ColumnTransformer` handles imputation, one-hot encoding, and scaling for every model — wrapping it in a `Pipeline` means `GridSearchCV` re-fits the preprocessor inside each CV fold (no leakage), and the final `rent_pipeline.pkl` accepts raw input.

**Preprocessing branches inside the `ColumnTransformer`:**
- *Numeric* (counts, distances, lat/lon, `n_amenities`): `SimpleImputer(median)` → `StandardScaler`
- *Categorical* (`neighbourhood_cleansed`, `room_type`): `OneHotEncoder(handle_unknown='ignore')`
- *Binary flags* (`has_*`): passthrough (already 0/1)

**Model selection:** 5-fold CV-R² on the **training set only**; the held-out test set is touched **exactly once**, on the winner.

In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import cross_val_score, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
import joblib

X_train = pd.read_csv('../data/processed/X_train.csv')
X_test  = pd.read_csv('../data/processed/X_test.csv')
y_train = pd.read_csv('../data/processed/y_train.csv').squeeze()
y_test  = pd.read_csv('../data/processed/y_test.csv').squeeze()
print(f'Train: {X_train.shape}, Test: {X_test.shape}')
print(f'Categorical columns: {X_train.select_dtypes(include="object").columns.tolist()}')

Train: (2052, 39), Test: (514, 39)
Categorical columns: ['neighbourhood_cleansed', 'room_type', 'property_type']


In [2]:
# Define the ColumnTransformer used by every candidate Pipeline.
CATEGORICAL = ['neighbourhood_cleansed', 'room_type', 'property_type']
BINARY      = ['has_ac', 'has_pool', 'has_hot_tub', 'has_gym',
               'has_dishwasher', 'has_washer', 'has_dryer',
               'has_free_parking', 'has_elevator', 'has_balcony',
               'host_is_superhost']
NUMERIC     = [c for c in X_train.columns if c not in CATEGORICAL + BINARY]

preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline([
            ('impute', SimpleImputer(strategy='median')),
            ('scale',  StandardScaler()),
        ]), NUMERIC),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), CATEGORICAL),
        ('bin', 'passthrough', BINARY),
    ],
    remainder='drop',
)

def make_pipeline(model):
    """Wrap any regressor in a Pipeline that runs the shared ColumnTransformer first."""
    return Pipeline([('pre', preprocessor), ('model', model)])

def cv_r2(pipe, X_tr, y_tr, cv=5):
    return cross_val_score(pipe, X_tr, y_tr, cv=cv, scoring='r2').mean()

# Dummy baseline — predicts the mean log-price for every listing.
dummy_pipe = make_pipeline(DummyRegressor(strategy='mean'))
dummy_cv   = cv_r2(dummy_pipe, X_train, y_train)
print(f'{"Dummy (mean)":24s} CV-R²(log) = {dummy_cv:+.3f}')
print(f'Numeric features    : {len(NUMERIC)}  → {NUMERIC}')
print(f'Categorical features: {len(CATEGORICAL)} → {CATEGORICAL}')
print(f'Binary flags        : {len(BINARY)}  → {BINARY}')

Dummy (mean)             CV-R²(log) = -0.003
Numeric features    : 25  → ['accommodates', 'bedrooms', 'bathrooms', 'minimum_nights', 'latitude', 'longitude', 'availability_365', 'number_of_reviews', 'reviews_per_month', 'n_amenities', 'host_response_rate', 'host_acceptance_rate', 'review_scores_rating', 'review_scores_accuracy', 'review_scores_cleanliness', 'review_scores_checkin', 'review_scores_communication', 'review_scores_location', 'review_scores_value', 'dist_hallenstadion_km', 'dist_letzigrund_km', 'dist_messe_km', 'dist_opernhaus_km', 'dist_hb_km', 'min_dist_venue_km']
Categorical features: 3 → ['neighbourhood_cleansed', 'room_type', 'property_type']
Binary flags        : 11  → ['has_ac', 'has_pool', 'has_hot_tub', 'has_gym', 'has_dishwasher', 'has_washer', 'has_dryer', 'has_free_parking', 'has_elevator', 'has_balcony', 'host_is_superhost']


In [3]:
# Linear Regression — no hyperparameters, just CV-score the pipeline.
lr_pipe = make_pipeline(LinearRegression())
lr_cv   = cv_r2(lr_pipe, X_train, y_train)
print(f'{"Linear Regression":24s} CV-R²(log) = {lr_cv:.3f}')

Linear Regression        CV-R²(log) = 0.451


In [4]:
# Random Forest — grid search over the model step inside the Pipeline.
# Param names are prefixed `model__` because the estimator is named 'model' in make_pipeline().
rf_pipe = make_pipeline(RandomForestRegressor(random_state=42, n_jobs=-1))
rf_grid = GridSearchCV(
    rf_pipe,
    param_grid={
        'model__n_estimators':     [200, 400],
        'model__max_depth':        [None, 15, 30],
        'model__min_samples_leaf': [1, 3],
    },
    cv=5, scoring='r2', n_jobs=-1, verbose=0,
)
rf_grid.fit(X_train, y_train)
rf_best = rf_grid.best_estimator_
rf_cv   = rf_grid.best_score_
print(f'{"Random Forest (tuned)":24s} CV-R²(log) = {rf_cv:.3f}')
print(f'  best params: {rf_grid.best_params_}')

Random Forest (tuned)    CV-R²(log) = 0.583
  best params: {'model__max_depth': 30, 'model__min_samples_leaf': 3, 'model__n_estimators': 400}


In [5]:
# HistGradientBoosting — same pattern.
hgb_pipe = make_pipeline(HistGradientBoostingRegressor(random_state=42))
hgb_grid = GridSearchCV(
    hgb_pipe,
    param_grid={
        'model__max_iter':         [200, 400],
        'model__max_depth':        [None, 6],
        'model__learning_rate':    [0.05, 0.1],
        'model__min_samples_leaf': [10, 20],
    },
    cv=5, scoring='r2', n_jobs=-1, verbose=0,
)
hgb_grid.fit(X_train, y_train)
hgb_best = hgb_grid.best_estimator_
hgb_cv   = hgb_grid.best_score_
print(f'{"HistGradientBoosting":24s} CV-R²(log) = {hgb_cv:.3f}')
print(f'  best params: {hgb_grid.best_params_}')

HistGradientBoosting     CV-R²(log) = 0.635
  best params: {'model__learning_rate': 0.05, 'model__max_depth': None, 'model__max_iter': 400, 'model__min_samples_leaf': 10}


## Model selection + final test evaluation

Pick the model with the highest CV-R². Fit it once on the full training set and evaluate **once** on the held-out test set. This is the only place the test set is touched in this notebook.

In [6]:
# All three candidates are sklearn Pipelines (preprocess → model), so the leaderboard
# and the final fit/predict work identically across them.
candidates = {
    'Linear Regression':     (lr_pipe,  lr_cv),
    'Random Forest (tuned)': (rf_best,  rf_cv),
    'HistGradientBoosting':  (hgb_best, hgb_cv),
}

leaderboard = pd.DataFrame(
    [(name, cv) for name, (_, cv) in candidates.items()],
    columns=['Model', 'CV-R²(log)']
).sort_values('CV-R²(log)', ascending=False).reset_index(drop=True)
print(leaderboard.to_string(index=False))

winner_name, (winner_model, winner_cv) = max(candidates.items(), key=lambda kv: kv[1][1])
print(f'\nWinner (by CV-R²): {winner_name}')

                Model  CV-R²(log)
 HistGradientBoosting    0.635357
Random Forest (tuned)    0.582765
    Linear Regression    0.451354

Winner (by CV-R²): HistGradientBoosting


In [7]:
# Fit winner on full train, evaluate ONCE on test. Save.
winner_model.fit(X_train, y_train)
pred_log = winner_model.predict(X_test)
pred     = np.expm1(pred_log)   # back to CHF
y_orig   = np.expm1(y_test)

mae  = mean_absolute_error(y_orig, pred)
rmse = np.sqrt(mean_squared_error(y_orig, pred))
r2   = r2_score(y_orig, pred)
print(f'{winner_name} — held-out test:')
print(f'  MAE   = {mae:.2f} CHF')
print(f'  RMSE  = {rmse:.2f} CHF')
print(f'  R²    = {r2:.3f}')

joblib.dump(winner_model, '../models/rent_pipeline.pkl')
print(f'\nSaved winner ({winner_name}) to models/rent_pipeline.pkl')

HistGradientBoosting — held-out test:
  MAE   = 38.80 CHF
  RMSE  = 67.02 CHF
  R²    = 0.530

Saved winner (HistGradientBoosting) to models/rent_pipeline.pkl
